# Classifier-Free Guidance

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Classifier-Free Guidance treina um único modelo de difusão condicional que também ignora a condição com certa probabilidade (virando incondicional). Na amostragem, extrapolamos da predição incondicional para a condicional, reforçando o alinhamento com o prompt.


## Formulação Matemática

$$\tilde \epsilon(x_t, c) = (1 + w)\,\epsilon_\theta(x_t, c) - w\,\epsilon_\theta(x_t, \varnothing)$$

A escala de guidance $w$ troca diversidade ($w = 0$) por fidelidade à condição ($w \gg 0$).


## Implementação


In [ ]:
import torch
import torch.nn as nn


In [ ]:
class ConditionalEpsilon(nn.Module):
    """Stub: a real model takes (x_t, t, c). Here we just demonstrate the math."""
    def __init__(self):
        super().__init__()
    def forward(self, x_t, t, c):
        # Simulate a conditional prediction with a deterministic offset
        return x_t + (0.0 if c is None else 0.5) * torch.ones_like(x_t)

def cfg_step(model, x_t, t, c, w=2.0, p_drop=0.1):
    eps_uncond = model(x_t, t, c=None)
    eps_cond   = model(x_t, t, c=c)
    return (1 + w) * eps_cond - w * eps_uncond


## Experimento


In [ ]:
model = ConditionalEpsilon()
x = torch.zeros(3, 4)
for w in [0.0, 1.0, 3.0, 7.5]:
    print(f'w={w:>4}: ', cfg_step(model, x, t=torch.tensor([0]), c='dog', w=w).mean().item())


## Discussão

- No treino, ignore a condição aleatoriamente (~10–20% dos batches) para a mesma rede rodar ambos os ramos.
- $w$ alto reforça a fidelidade ao prompt mas perde diversidade e pode saturar amostras.
- Quase todos os modelos modernos text-to-image (Stable Diffusion, Imagen, DALL·E 3) usam CFG com $w \in [1, 12]$.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
